This lab is how to use Pandas to clean and analyze search term data, I removed punctuation, numbers, corrected spelling, counted the frequency of search terms, and compared the performance and memory usage of Python and Pandas

In [1]:
# In this cell: Import libraries and declare your cleaning functions.
# These functions can be directly copied from Lab 4 if they were done appropriately. 
# These functions must take in a token and return a cleaned version of the token.
import pandas as pd
import re
import time
import sys

from spellchecker import SpellChecker
spell = SpellChecker()
def remove_punctuation(token):
    return re.sub(r'[^\w\s]', '', token)
def remove_numbers(token):
    return re.sub(r'\d', '', token)
def spell_correct(token):
    if token == "":
        return ""
    corrected = spell.correction(token)
    if corrected is None:
        return token
    return corrected

In [2]:
# In this cell: import the dataset, split into individual tokens, create DataFrame, 
# and display the first 20 lines of the DataFrame.
data = pd.read_csv(r"C:\Users\huelkes\Downloads\searchTerms.csv")
tokens = []

for search in data["SearchTerm"].dropna():
    search = str(search)
    search = search.replace("%20", " ")
    
    words = search.split()
    
    for word in words:
        tokens.append(word)

df = pd.DataFrame(tokens, columns=["original_terms"])

df.head(20)

,original_terms
0,36969
1,CMED
2,500100
3,KEND
4,5750
5,CMED
6,980228
7,DYNC1815H
8,DYND70642
9,DEES


In [3]:
# In this cell: Add a new columns to the DataFrame by applying your cleaning 
# functions one at a time. You may break this step into multiple cells if it 
# makes your benchmarking easier. Display the first 20 lines of the DataFrame 
# once it has the four required columns.
start = time.time()

df["punctuation_removed"] = df["original_terms"].map(remove_punctuation)

end = time.time()

print("Punctuation removal time:", end - start, "seconds")

Punctuation removal time: 5.1248109340667725 seconds


In [4]:
start = time.time()

df["number_removed"] = df["punctuation_removed"].map(remove_numbers)

end = time.time()

print("Number removal time:", end - start, "seconds")

Number removal time: 5.695549726486206 seconds


In [5]:
start = time.time()

# Make spell checking much faster
spell.distance = 1

unique_words = set(df["number_removed"])

# Only check actual words of reasonable length
words_to_check = [
    word for word in unique_words
    if word.isalpha() and 2 < len(word) < 20
]

misspelled = spell.unknown(words_to_check)

spell_dictionary = {}

for word in misspelled:
    corrected = spell.correction(word)

    if corrected is not None:
        spell_dictionary[word] = corrected

df["spell_corrected"] = df["number_removed"].map(
    lambda word: spell_dictionary.get(word, word)
)

end = time.time()

print("Spell correction time:", end - start, "seconds")
print("Words checked:", len(words_to_check))

df.head(20)

Spell correction time: 19.29267144203186 seconds
Words checked: 20011


,original_terms,punctuation_removed,number_removed,spell_corrected
0,36969,36969,,
1,CMED,CMED,CMED,CMED
2,500100,500100,,
3,KEND,KEND,KEND,KEND
4,5750,5750,,
5,CMED,CMED,CMED,CMED
6,980228,980228,,
7,DYNC1815H,DYNC1815H,DYNCH,DYNCH
8,DYND70642,DYND70642,DYND,DYND
9,DEES,DEES,DEES,DEES


In [6]:
# In this cell: export most clean column to list, count terms with Lab 4 dictionary 
# counting code, display benchmarking results.
clean_list = df["spell_corrected"].tolist()

start = time.time()

frequency = {}

for word in clean_list:
    if word in frequency:
        frequency[word] += 1
    else:
        frequency[word] = 1

sorted_frequency = sorted(
    frequency.items(),
    key=lambda item: item[1],
    reverse=True
)

end = time.time()

python_time = end - start

print("Python dictionary time:", python_time, "seconds")

print(sorted_frequency[:20])

Python dictionary time: 1.0190000534057617 seconds
[('', 164579), ('chicken', 18032), ('cream', 15226), ('cheese', 13344), ('beef', 12882), ('juice', 10959), ('pie', 10944), ('sauce', 10686), ('pork', 10376), ('potato', 10054), ('green', 9103), ('diced', 8988), ('beans', 8624), ('tomato', 8355), ('corn', 7684), ('bread', 7352), ('apple', 7304), ('sausage', 7002), ('mix', 6720), ('cake', 6700)]


In [7]:
# In this cell: use .value_counts() to do frequency counting. Display benchmarking
# results.
start = time.time()

pandas_frequency = df["spell_corrected"].value_counts()

end = time.time()

pandas_time = end - start

print("Pandas value_counts time:", pandas_time, "seconds")

print(pandas_frequency.head(20))

Pandas value_counts time: 0.17026519775390625 seconds
spell_corrected
           164579
chicken     18032
cream       15226
cheese      13344
beef        12882
juice       10959
pie         10944
sauce       10686
pork        10376
potato      10054
green        9103
diced        8988
beans        8624
tomato       8355
corn         7684
bread        7352
apple        7304
sausage      7002
mix          6720
cake         6700
Name: count, dtype: int64


**TODO:** Analyze above benchmarking results.

In [8]:
# In this cell: Evaluate space efficiency of python list vs pandas DataFrame.
# Display results. Again, you may break this step into multiple cells if it 
# makes your analysis easier.
# Pandas memory

pandas_memory = df["spell_corrected"].memory_usage(deep=True)

print("Pandas memory:", pandas_memory, "bytes")


# Python list memory

list_memory = sys.getsizeof(clean_list)

for word in clean_list:
    list_memory += sys.getsizeof(word)

print("Python list memory:", list_memory, "bytes")

Pandas memory: 19009654 bytes
Python list memory: 79252278 bytes


**TODO:** Analyze above space analysis results.

In [9]:
# additional to make it look nice
print("Pandas memory:", pandas_memory / 1024 / 1024, "MB")
print("Python list memory:", list_memory / 1024 / 1024, "MB")

Pandas memory: 18.129018783569336 MB
Python list memory: 75.58086204528809 MB
